# 03 实验数据质量与聚合推断检查

- **SRM**：以分流单位 cookie 的计数 Pageviews 检验 Control/Experiment 是否符合预期 50:50（χ² + 精确二项）。
- **Invariant Metrics**：Pageviews、Clicks、CTP 这些在筛选步骤之前发生、本不应被 treatment 影响的指标是否异常。
- **日粒度趋势与组间一致性**：时间序列、逐日差值、分布与异常日期——**属于实验健康/数据诊断，不称为标准 covariate balance test**。
- 口径：SRM 与 invariant 全部使用锁定的**全 37 天流量窗（2014-10-11→11-16）**。
- 两项可选鲁棒性检查不在本 Notebook 内。

In [1]:
# 加载配置与长表（全 37 天流量窗）
import json
import tomllib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def _weekly_ticks(fig, *axes):
    # 周级刻度，避免日标签重叠
    for ax in axes:
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.SU))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    fig.autofmt_xdate(rotation=0, ha="center")

ROOT = Path.cwd()
with open(ROOT / "config" / "analysis_config.toml", "rb") as f:
    CFG = tomllib.load(f)
long = pd.read_csv(ROOT / "data/processed/daily_long.csv", parse_dates=["Date"]).sort_values(["Date", "Group"])
FIG = ROOT / "reports" / "figures"
traffic = long.copy()  # SRM/invariant 用全 37 天
print("traffic window rows:", len(traffic), "| days:", traffic["Date"].nunique())
tot = traffic.groupby("Group")[["Pageviews", "Clicks"]].sum()
tot

traffic window rows: 74 | days: 37


,Pageviews,Clicks
Group,,
Control,345543,28378
Experiment,344660,28325


## SRM（Sample Ratio Mismatch）
预期分流比例 50:50（cookie 级随机分流，聚合到日不改变期望比例）。同时给 χ² 拟合优度与精确二项检验，并报实验组分到的比例及其 95% CI。

In [2]:
# SRM on Pageviews（分流单位计数）
pv_c = int(tot.loc["Control", "Pageviews"]); pv_e = int(tot.loc["Experiment", "Pageviews"])
n_pv = pv_c + pv_e
expected_ratio = np.array([0.5, 0.5])
obs = np.array([pv_c, pv_e]); exp = expected_ratio * n_pv
chi2 = ((obs - exp) ** 2 / exp).sum()
p_chi2 = stats.chi2.sf(chi2, df=1)
bt = stats.binomtest(pv_e, n_pv, 0.5)           # 精确二项（Experiment 计数）
p_binom = bt.pvalue
ci_low, ci_high = bt.proportion_ci(confidence_level=0.95)
srm = {
    "metric": "Pageviews", "control": pv_c, "experiment": pv_e, "total": n_pv,
    "expected_ratio": "50:50",
    "experiment_share": pv_e / n_pv,
    "chi2": float(chi2), "p_chi2": float(p_chi2),
    "p_binom_exact": float(p_binom),
    "share_95CI": [float(ci_low), float(ci_high)],
}
print(f"Pageviews: C={pv_c:,}, E={pv_e:,}, total={n_pv:,}")
print(f"experiment share={pv_e/n_pv:.6f}, 95%CI=({ci_low:.6f},{ci_high:.6f})  [0.5 in CI]")
print(f"chi2={chi2:.4f} (df=1), p_chi2={p_chi2:.4f}; exact binomial two-sided p={p_binom:.4f}")
print("SRM 判定：", "未发现分流比例失配（p>>0.05，且 0.5 落在份额 CI 内）" if p_chi2 > 0.05 and ci_low <= 0.5 <= ci_high else "存在 SRM 风险，需排查")

Pageviews: C=345,543, E=344,660, total=690,203
experiment share=0.499360, 95%CI=(0.498180,0.500541)  [0.5 in CI]
chi2=1.1297 (df=1), p_chi2=0.2878; exact binomial two-sided p=0.2884
SRM 判定： 未发现分流比例失配（p>>0.05，且 0.5 落在份额 CI 内）


## Invariant Metrics（不应被 treatment 改变的指标）
1. **Clicks 分流**：点击发生在筛选步骤之前，两组点击量应≈50:50（χ² + 精确二项）。
2. **CTP = Clicks/Pageviews**：两比例比较（z 检验用 pooled 方差，差值 CI 用 unpooled），给效应量与 CI，而非只看 p。
Pageviews 的均衡已由 SRM 覆盖。

In [3]:
# 20.1 Clicks 分流均衡
ck_c = int(tot.loc["Control", "Clicks"]); ck_e = int(tot.loc["Experiment", "Clicks"]); n_ck = ck_c+ck_e
obs_ck = np.array([ck_c, ck_e]); exp_ck = np.array([n_ck/2, n_ck/2])
chi2_ck = ((obs_ck-exp_ck)**2/exp_ck).sum(); p_chi2_ck = stats.chi2.sf(chi2_ck, 1)
bt_ck = stats.binomtest(ck_e, n_ck, 0.5); p_binom_ck = bt_ck.pvalue
cil, cih = bt_ck.proportion_ci(.95)
print(f"Clicks: C={ck_c:,}, E={ck_e:,}, E share={ck_e/n_ck:.6f} 95%CI=({cil:.6f},{cih:.6f})")
print(f"chi2={chi2_ck:.4f}, p_chi2={p_chi2_ck:.4f}; exact binomial p={p_binom_ck:.4f}")

# 20.2 CTP 两比例比较（全 37 天）
ctp_c, ctp_e = ck_c/pv_c, ck_e/pv_e
p_pool = (ck_c+ck_e)/(pv_c+pv_e)
se_pool = np.sqrt(p_pool*(1-p_pool)*(1/pv_c + 1/pv_e))
z_ctp = (ctp_e-ctp_c)/se_pool; p_ctp = 2*stats.norm.sf(abs(z_ctp))
se_un = np.sqrt(ctp_c*(1-ctp_c)/pv_c + ctp_e*(1-ctp_e)/pv_e)
hw = stats.norm.ppf(.975)*se_un
print(f"\nCTP: C={ctp_c:.6f}, E={ctp_e:.6f}, diff(E-C)={ctp_e-ctp_c:+.6f}")
print(f"z={z_ctp:.4f}, p={p_ctp:.4f}; diff 95%CI=({ctp_e-ctp_c-hw:+.6f},{ctp_e-ctp_c+hw:+.6f})")

invariants = {
    "Clicks_split": {"control": ck_c, "experiment": ck_e, "experiment_share": ck_e/n_ck,
                     "share_95CI": [float(cil), float(cih)],
                     "chi2": float(chi2_ck), "p_chi2": float(p_chi2_ck), "p_binom_exact": float(p_binom_ck)},
    "CTP": {"control": ctp_c, "experiment": ctp_e, "diff": ctp_e-ctp_c, "z": float(z_ctp),
            "p": float(p_ctp), "diff_95CI": [float(ctp_e-ctp_c-hw), float(ctp_e-ctp_c+hw)]},
}

Clicks: C=28,378, E=28,325, E share=0.499533 95%CI=(0.495409,0.503657)
chi2=0.0495, p_chi2=0.8239; exact binomial p=0.8271

CTP: C=0.082126, E=0.082182, diff(E-C)=+0.000057
z=0.0857, p=0.9317; diff 95%CI=(-0.001239,+0.001352)


## 日粒度趋势与组间一致性（实验健康诊断，非 covariate balance test）
同一自然日两组并排：看 Pageviews、CTP 的时间序列与逐日差值（E−C）是否稳定同步、有无系统性偏移或异常日。配对日级 t/Wilcoxon 仅作诊断（n=37，聚合数据，不替代 cookie 级判断）。

In [4]:
# 组装逐日宽表（流量窗指标）
daily = traffic.pivot(index="Date", columns="Group")
w = pd.DataFrame({
    "Weekday": traffic.pivot(index="Date", columns="Group")["Weekday"]["Control"],
    "PV_C": daily["Pageviews"]["Control"], "PV_E": daily["Pageviews"]["Experiment"],
    "CK_C": daily["Clicks"]["Control"], "CK_E": daily["Clicks"]["Experiment"],
})
w["CTP_C"] = w["CK_C"]/w["PV_C"]; w["CTP_E"] = w["CK_E"]/w["PV_E"]
w["PV_diff"] = w["PV_E"]-w["PV_C"]; w["CTP_diff"] = w["CTP_E"]-w["CTP_C"]
w["PV_rel_diff"] = w["PV_diff"]/w["PV_C"]
print("逐日差值描述：")
print(w[["PV_diff","PV_rel_diff","CTP_diff"]].describe().round(6).to_string())

t_pv = stats.ttest_rel(w["PV_E"], w["PV_C"]); w_pv = stats.wilcoxon(w["PV_E"], w["PV_C"])
t_ctp = stats.ttest_rel(w["CTP_E"], w["CTP_C"]); w_ctp = stats.wilcoxon(w["CTP_E"], w["CTP_C"])
print(f"\n配对日级诊断 PV  : mean diff={w['PV_diff'].mean():.1f}, paired t p={t_pv.pvalue:.4f}, Wilcoxon p={w_pv.pvalue:.4f}")
print(f"配对日级诊断 CTP : mean diff={w['CTP_diff'].mean():.6f}, paired t p={t_ctp.pvalue:.4f}, Wilcoxon p={w_ctp.pvalue:.4f}")
# 周末季节性与异常日（沿用|z|>=2.5 口径）
for col in ["PV_C","PV_E","CTP_C","CTP_E"]:
    z = (w[col]-w[col].mean())/w[col].std(ddof=1)
    hit = w.index[z.abs() >= 2.5]
    for d in hit: print(f"极端日 {col} {d.date()} ({w.loc[d,'Weekday']}) value={w.loc[d,col]:.5f} z={z[d]:+.2f}")

逐日差值描述：
          PV_diff  PV_rel_diff   CTP_diff
count   37.000000    37.000000  37.000000
mean   -23.864865    -0.002232   0.000061
std    102.962281     0.011319   0.001843
min   -221.000000    -0.022069  -0.004093
25%    -79.000000    -0.008265  -0.001367
50%    -15.000000    -0.001686  -0.000049
75%     33.000000     0.003525   0.001220
max    230.000000     0.030939   0.003508

配对日级诊断 PV  : mean diff=-23.9, paired t p=0.1672, Wilcoxon p=0.1869
配对日级诊断 CTP : mean diff=0.000061, paired t p=0.8409, Wilcoxon p=0.8580
极端日 PV_C 2014-10-18 (Sat) value=7434.00000 z=-2.57
极端日 CTP_C 2014-10-24 (Fri) value=0.07134 z=-3.34
极端日 CTP_E 2014-10-24 (Fri) value=0.07413 z=-2.59


In [5]:
# 图3a：Pageviews 时间序列 + 逐日差值
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), dpi=150, sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(w.index, w["PV_C"], label="Control", color="#1f77b4")
ax1.plot(w.index, w["PV_E"], label="Experiment", color="#d62728", ls="--")
ax1.set_ylabel("Daily Pageviews"); ax1.set_title("Daily Pageviews by group (37-day traffic window)")
ax1.legend(); ax1.grid(alpha=.3)
colors = ["#d62728" if x < 0 else "#1f77b4" for x in w["PV_diff"]]
ax2.bar(w.index, w["PV_diff"], color=colors, width=1.0)
ax2.axhline(0, color="black", lw=.8); ax2.set_ylabel("E − C Pageviews"); ax2.grid(alpha=.3)
_weekly_ticks(fig, ax1, ax2)
fig.tight_layout(); fig.savefig(FIG / "fig_pageviews_ts.png"); plt.close(fig)

# 图3b：CTP 时间序列 + 逐日差值
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), dpi=150, sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(w.index, w["CTP_C"], label="Control", color="#1f77b4")
ax1.plot(w.index, w["CTP_E"], label="Experiment", color="#d62728", ls="--")
ax1.set_ylabel("Daily CTP"); ax1.set_title("Daily Click-Through-Probability by group")
ax1.legend(); ax1.grid(alpha=.3)
ax2.bar(w.index, w["CTP_diff"], color="gray", width=1.0)
ax2.axhline(0, color="black", lw=.8); ax2.set_ylabel("E − C CTP"); ax2.grid(alpha=.3)
_weekly_ticks(fig, ax1, ax2)
fig.tight_layout(); fig.savefig(FIG / "fig_ctp_ts.png"); plt.close(fig)
print("saved fig_pageviews_ts.png, fig_ctp_ts.png")

saved fig_pageviews_ts.png, fig_ctp_ts.png


In [6]:
# 落盘机器可读质量检查结果 + 回读验证
quality = {"window": "37-day traffic window (2014-10-11..2014-11-16)",
           "SRM": srm, "invariants": invariants,
           "daily_diag": {"PV_mean_daily_diff": float(w["PV_diff"].mean()),
                          "PV_paired_t_p": float(t_pv.pvalue), "PV_wilcoxon_p": float(w_pv.pvalue),
                          "CTP_mean_daily_diff": float(w["CTP_diff"].mean()),
                          "CTP_paired_t_p": float(t_ctp.pvalue), "CTP_wilcoxon_p": float(w_ctp.pvalue)}}
outp = ROOT/"data"/"processed"/"quality_checks.json"
outp.write_text(json.dumps(quality, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
back = json.loads(outp.read_text(encoding="utf-8"))
assert abs(back["SRM"]["p_chi2"] - srm["p_chi2"]) < 1e-15
print("quality_checks.json written & re-read OK")

quality_checks.json written & re-read OK


## 小结
1. **SRM 未失配**：Pageviews 两组≈50:50，χ² 与精确二项 p 值均远大于 0.05，实验组分到份额的 95% CI 包含 0.5。
2. **Invariant metrics 无异常**：Clicks 分流同样均衡；CTP 两组点值几乎相同，差值 95% CI 跨 0 且量级可忽略——没有证据表明筛选步骤反向影响了筛选之前的行为。
3. **日粒度同步**：两组 PV/CTP 走势一致、周末同形态季节性；逐日差值无系统性偏移；10/18、10/24 为两组共同波动，不构成组间失衡。
4. 以上为聚合数据的实验健康诊断；cookie 级 AA 因无用户级数据无法开展，日粒度 permutation待批准后再决定是否作为补充鲁棒性检查。